In [1]:
# 1. Manipulação e Análise de Dados
import pandas as pd
import numpy as np
# 2. Visualização de Dados (Foco em Interatividade)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# 3. Pré-processamento e Divisão de Dados
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
# 4. Modelagem
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
# 5. Métricas de Avaliação (Foco em Saúde/Healthcare)
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve, 
    f1_score
)
# 6. Interpretabilidade do Modelo
import shap

import ipywidgets as widgets
from IPython.display import display, clear_output


In [2]:
df = pd.read_csv('heart_failure_clinical_records_dataset.csv')
df_traduzido_pt_br = df.rename(columns={
    'age' : "Idade",
    'anaemia' : 'Anemia',
    'creatinine_phosphokinase' : 'Creatina Fosfoquinase',
    'ejection_fraction' : 'Fração de Ejeção',
    'high_blood_pressure' : 'Pressão Alta',
    'platelets' : 'Plaquetas',
    'serum_creatinine' : 'Creatinina Sérica',
    'serum_sodium' : 'Sódio Sérico',
    'sex' : 'Sexo',
    'diabetes' : 'Diabetes',
    'smoking' : 'Fumante',
    'time' : 'Tempo',
    'DEATH_EVENT' : 'Evento de Óbito'
})
df_traduzido_pt_br.head()

,Idade,Anemia,Creatina Fosfoquinase,Diabetes,Fração de Ejeção,Pressão Alta,Plaquetas,Creatinina Sérica,Sódio Sérico,Sexo,Fumante,Tempo,Evento de Óbito
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [3]:
# 1. Lista todas as colunas do DataFrame
print("Colunas encontradas no dataset:")
print(df.columns.tolist())

# 2. Verifica se há espaços em branco invisíveis (comum em CSVs)
# Exemplo: ' DEATH_EVENT' em vez de 'DEATH_EVENT'
print("\nVerificando espaços extras:")
print([f"'{col}'" for col in df.columns])

# 3. DICA DE OURO: Limpeza automática de nomes de colunas
# Remove espaços no início/fim e garante que o código não quebre por bobeira
df.columns = df.columns.str.strip()
print("\nColunas após o strip (limpeza):")
print(df.columns.tolist())

Colunas encontradas no dataset:
['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT']

Verificando espaços extras:
["'age'", "'anaemia'", "'creatinine_phosphokinase'", "'diabetes'", "'ejection_fraction'", "'high_blood_pressure'", "'platelets'", "'serum_creatinine'", "'serum_sodium'", "'sex'", "'smoking'", "'time'", "'DEATH_EVENT'"]

Colunas após o strip (limpeza):
['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT']


In [4]:
# Criando um dicionário
dados_explicativos = {
    'Coluna': [
        'Tempo', 'Creatinina Sérica', 'Fração de Ejeção', 'Idade', 
        'Sódio Sérico', 'CPK (Enzima)', 'Plaquetas', 'Pressão Alta', 
        'Anemia', 'Diabetes', 'Fumante', 'Sexo'
    ],
    'O que representa na Saúde?': [
        'Dias de acompanhamento médico do paciente.',
        'Nível de resíduo no sangue; indica se os Rins estão funcionando bem.',
        'Porcentagem de sangue que o coração consegue bombear (força do coração).',
        'Idade cronológica do paciente.',
        'Equilíbrio de sais minerais e hidratação no corpo.',
        'Proteína que aumenta no sangue quando há lesão no músculo do coração.',
        'Células que ajudam na coagulação do sangue.',
        'Se o paciente possui diagnóstico de hipertensão.',
        'Redução de glóbulos vermelhos, o que diminui o transporte de oxigênio.',
        'Presença de níveis elevados de açúcar no sangue.',
        'Se o paciente possui o hábito de fumar.',
        'Gênero biológico do paciente.'
    ]
}

# Transformando em um DataFrame para exibição bonita
df_ajuda = pd.DataFrame(dados_explicativos)


In [5]:
X = df_traduzido_pt_br.drop('Evento de Óbito', axis=1)
y = df_traduzido_pt_br['Evento de Óbito']

# Divisão Treino/Teste (essencial para não viciar o modelo)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Escalonamento (Uso do StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dados prontos! Treino: {X_train.shape[0]} amostras, Teste: {X_test.shape[0]} amostras.")

Dados prontos! Treino: 209 amostras, Teste: 90 amostras.


In [6]:
# Treinando o XGBoost 
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# Treinando a RandomForest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

print("Modelos treinados com sucesso!")

Modelos treinados com sucesso!


In [7]:
# 1. Definindo X (características) e y (alvo) usando os nomes traduzidos
# Note que 'diabetes' não estava no seu rename, então ele permanece como 'diabetes'
X = df_traduzido_pt_br.drop('Evento de Óbito', axis=1)
y = df_traduzido_pt_br['Evento de Óbito']

# 2. Divisão Treino/Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Escalonamento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Treinamento do Modelo (XGBoost)
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# 5. Visualização do Desempenho (Matriz de Confusão com Plotly)
y_pred = xgb_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred)

fig = px.imshow(cm, text_auto=True, 
                x=['Previsão: Sobreviveu', 'Previsão: Óbito'], 
                y=['Real: Sobreviveu', 'Real: Óbito'],
                color_continuous_scale='RdBu_r',
                title="<b>Matriz de Confusão (Dados Traduzidos)</b>")

In [8]:
# Extraindo a importância do modelo
importancias = xgb_model.feature_importances_
feature_names = X.columns

# Criando um DataFrame temporário para o gráfico
df_importancia = pd.DataFrame({'Atributo': feature_names, 'Importancia': importancias})
df_importancia = df_importancia.sort_values(by='Importancia', ascending=True)

# Gerando o gráfico de barras horizontais
fig = px.bar(df_importancia, x='Importancia', y='Atributo', orientation='h',
             title="<b>Ranking de Relevância Clínica</b>",
             labels={'Importancia': 'Poder de Predição', 'Atributo': 'Exame Médico'},
             color='Importancia', color_continuous_scale='Viridis')


In [9]:
def analise_interativa_de_risco(dados_paciente):
    # 1. Preparação e Predição
    novo_df = pd.DataFrame([dados_paciente])
    novo_df_scaled = scaler.transform(novo_df)
    
    prob_atual = xgb_model.predict_proba(novo_df_scaled)[0][1]
    
    print(f"===  RELATÓRIO DE INTERVENÇÃO CLÍNICA ===")
    print(f"Risco de óbito calculado: {prob_atual:.2%}")
    print("-" * 45)
    print("ANÁLISE DE FATORES E RECOMENDAÇÕES:\n")

    # 2. Diagnóstico por variáveis
    
    # FRAÇÃO DE EJEÇÃO (Normalmente acima de 50-55%)
    if dados_paciente['Fração de Ejeção'] < 40:
        print(f"ALERTA: Fração de Ejeção baixa ({dados_paciente['Fração de Ejeção']}%).")
        print("-> Ação: Fortalecer o bombeamento cardíaco através de medicação ou terapia.\n")
    
    # CREATININA SÉRICA (Normalmente entre 0.7 e 1.3 mg/dL)
    if dados_paciente['Creatinina Sérica'] > 1.3:
        print(f"ALERTA: Creatinina elevada ({dados_paciente['Creatinina Sérica']:.2f} mg/dL).")
        print("-> Ação: Monitorar função renal. Hidratação e revisão de medicamentos nefrotóxicos.\n")
    
    # SÓDIO SÉRICO (Normal entre 135-145 mEq/L)
    if dados_paciente['Sódio Sérico'] < 135:
        print(f"ALERTA: Hiponatremia detectada ({dados_paciente['Sódio Sérico']} mEq/L).")
        print("-> Ação: Reposição de eletrólitos e balanço hídrico.\n")

    # TABAGISMO
    if dados_paciente['Fumante'] == 1:
        print(" ALERTA: Paciente fumante ativo.")
        print("   -> Ação: Cessação tabágica imediata. Reduziria o estresse oxidativo e inflamação.\n")

    # TEMPO DE ACOMPANHAMENTO (Variável crítica no seu SHAP)
    if dados_paciente['Tempo'] < 50:
        print(f"OBSERVAÇÃO: Tempo de acompanhamento curto ({dados_paciente['Tempo']} dias).")
        print("-> Ação: Intensificar consultas no primeiro trimestre pós-evento.\n")

    # 3. Simulação de "Cenário Ideal"
    # Criamos um clone do paciente "curado" para mostrar a queda no risco
    paciente_ideal = dados_paciente.copy()
    paciente_ideal['Fumante'] = 0
    paciente_ideal['Creatinina Sérica'] = 1.0
    paciente_ideal['Fração de Ejeção'] = 50
    paciente_ideal['Sódio Sérico'] = 140
    paciente_ideal['Tempo'] = 100
    paciente_ideal['Plaquetas'] = 250.000
    paciente_ideal['Pressão Alta'] = 0
    paciente_ideal['Creatina Fosfoquinase'] = 150
    paciente_ideal['Anemia'] = 0
    
    ideal_scaled = scaler.transform(pd.DataFrame([paciente_ideal]))
    prob_ideal = xgb_model.predict_proba(ideal_scaled)[0][1]
    
    print("-" * 45)
    print(f" IMPACTO DAS MUDANÇAS:")
    print(f"Se estabilizarmos os fatores acima, o risco cai de {prob_atual:.2%} para {prob_ideal:.2%}.")
    print(f"Uma redução real de {(prob_atual - prob_ideal):.2%} na chance de óbito.")

# Teste com o paciente de risco
# analise_interativa_de_risco(exemplo_paciente)

In [10]:
# 1. CSS E IDENTIDADE VISUAL
css_saude = """
<style>
    @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@400;600&display=swap');
    .dashboard-main { font-family: 'Montserrat', sans-serif; }
    
    .widget-dropdown > select {
        background-color: #27ae60 !important;
        color: white !important;
        font-weight: 600 !important;
        font-size: 15px !important;
        border-radius: 12px !important;
        height: 45px !important;
        line-height: 35px !important;
        padding: 0 15px !important;
        border: none !important;
        box-shadow: 0 4px 10px rgba(0,0,0,0.1) !important;
        cursor: pointer;
    }

    .widget-label {
        min-width: 140px !important;
        font-family: 'Montserrat', sans-serif !important;
        font-weight: bold !important;
        color: #2c3e50 !important;
        font-size: 14px !important;
    }

    .tabela-guia th { background-color: #219150; color: white; padding: 12px; text-align: left; }
    .tabela-guia td { padding: 10px; border-bottom: 1px solid #eee; font-size: 14px; }
</style>
"""

header = widgets.HTML(
    "<div style='background: linear-gradient(135deg, #1e8449, #27ae60); color: white; padding: 25px; border-radius: 15px; margin-bottom: 10px;'>"
    "  <h2 style='margin:0;'> Estudo de Registros Clínicos de Insuficiência Cardíaca </h2>"
    "  <p style='margin:5px 0 0 0; opacity: 0.9;'>Análise de Relevância e Glossário de Saúde</p>"
    "</div>"
)

seletor = widgets.ToggleButtons(
    options=[
        ('🏆 Ranking de Relevância', 'ranking'),
        ('📈 Distribuição Etária', 'dist'),
        ('🔍 Matriz de Precisão', 'cm')
    ],
    value='ranking',
    description='VISUALIZAÇÃO:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='800px')
)

output_plot = widgets.Output(layout=widgets.Layout(
    width='100%',
    min_height='650px', 
    height='auto',      
    margin='5px 0 5px 0'
))

guia_html = widgets.HTML(
    "<div style='margin-top: 30px; border-top: 2px solid #27ae60; padding-top: 20px;'>"
    "<h3 style='color: #1e8449; font-family: Montserrat;'> Guia de Referência de Atributos</h3>" +
    df_ajuda.style.set_table_attributes('class="tabela-guia" style="width:100%;"')
    .set_properties(**{'text-align': 'left'})
    .hide(axis='index').to_html() +
    "</div>"
)

def atualizar(change):
    with output_plot:
        clear_output(wait=True)
        escolha = seletor.value
        
        if escolha == 'ranking':
            fig = px.bar(df_importancia, x='Importancia', y='Atributo', orientation='h',
                         title="<b>Ranking de Relevância Clínica</b>",
                         labels={'Importancia': 'Poder de Predição', 'Atributo': 'Exame Médico'},
                         color='Importancia', color_continuous_scale='Viridis')
            
            fig.update_layout(font_family="Montserrat", title_x=0.5, height=550)
            fig.show()
            
        elif escolha == 'dist':
            fig = px.histogram(df_traduzido_pt_br, x="Idade", color="Evento de Óbito",
                               marginal="box", title="Distribuição de Risco por Faixa Etária",
                               color_discrete_map={0: '#27ae60', 1: '#c0392b'})
            fig.update_layout(font_family="Montserrat", title_x=0.5, height=550)
            fig.show()
            
        elif escolha == 'cm':
            fig = px.imshow(cm, text_auto=True, x=['Vivo', 'Óbito'], y=['Real: Vivo', 'Real: Óbito'],
                            color_continuous_scale='Greens', title="Matriz de Confusão")
            fig.update_layout(font_family="Montserrat", title_x=0.5, height=550)
            fig.show()

seletor.observe(atualizar, names='value')

dashboard = widgets.VBox([
    widgets.HTML(css_saude),
    header,
    widgets.Box([seletor], layout=widgets.Layout(
        justify_content='center', 
        margin='20px 0 80px 0' 
    )),
    output_plot,
    guia_html
], layout=widgets.Layout(
    padding='20px',
    height='auto',
    overflow='visible'
))

display(dashboard)
atualizar(None)

# Análise de Determinantes de Sobrevivência
> **Projeto:** Sistema de Apoio à Decisão Clínica |

Com base no processamento do modelo **XGBoost**, os fatores abaixo foram identificados como os maiores determinantes estatísticos para garantir a **sobrevivência** e a estabilidade clínica do paciente:

---

### 1. Tempo de Acompanhamento (Estabilidade)
* **Relevância:** É o preditor número 1 de sobrevivência no conjunto de dados.
* **Impacto Clínico:** Pacientes que mantêm um longo histórico de monitoramento sem crises agudas demonstram maior resiliência biológica e melhor adaptação ao tratamento contínuo.

### 2.  Fração de Ejeção (Reserva Cardíaca)
* **Relevância:** Fundamental para a manutenção da perfusão sistêmica e oxigenação dos órgãos.
* **Impacto Clínico:** Manter a força de bombeamento do coração em níveis saudáveis (acima de **50-55%**) evita a falência de outros órgãos e garante a estabilidade hemodinâmica.

### 3.  Creatinina Sérica (Equilíbrio Cardiorrenal)
* **Relevância:** Principal indicador da saúde renal e da capacidade de filtração do organismo.
* **Impacto Clínico:** Valores baixos e estáveis (abaixo de **1.1 mg/dL**) impedem o acúmulo de toxinas e líquidos, protegendo o coração de sobrecargas volumétricas que podem ser fatais.

---

> ###  Conclusão do Modelo
> A sobrevivência é maximizada quando há uma sinergia entre a **força mecânica do coração** (Fração de Ejeção) e a **capacidade de filtração renal** (Creatinina), sustentadas por um acompanhamento clínico contínuo (**Tempo**).

In [12]:
css_clinico = """
<style>
    /* Botão Verde Clínico personalizado */
    .btn-analisar {
        background-color: #27ae60 !important;
        color: white !important;
        font-weight: bold !important;
        border-radius: 8px !important;
        transition: 0.3s !important;
    }
    .btn-analisar:hover {
        background-color: #1e8449 !important;
        box-shadow: 0 4px 8px rgba(0,0,0,0.2) !important;
    }
    /* Alinhamento dos rótulos dos widgets */
    .widget-label { 
        text-align: left !important; 
        font-weight: 600 !important;
        color: #34495e !important;
    }
</style>
"""

header = widgets.HTML(
    "<h1 style='text-align: center; color: #2c3e50; font-family: Montserrat;'>  PAINEL DE CONTROLE CLÍNICO </h1>"
    "<p style='text-align: center; color: #7f8c8d;'>Sistema de Apoio à Decisão Clínica - Engenharia de Computação UFG</p>"
    "<hr style='border: 1px solid #27ae60;'>"
)

# Estilo para alinhar as descrições à esquerda e definir largura
style = {'description_width': '160px'} 
layout_input = widgets.Layout(width='95%', margin='10px 0')

inputs = {
    'Idade': widgets.IntText(value=60, min=40, max=95, description='Idade:', style=style, layout=layout_input),
    'Anemia': widgets.ToggleButtons(options=[('Não', 0), ('Sim', 1)], description='Anemia:', style=style, layout=layout_input),
    'Creatina Fosfoquinase': widgets.IntText(value=250, description='Creatina Fosfoquinase:', style=style, layout=layout_input),
    'Diabetes': widgets.ToggleButtons(options=[('Não', 0), ('Sim', 1)], description='Diabetes:', style=style, layout=layout_input),
    'Fração de Ejeção': widgets.IntSlider(value=38, min=10, max=80, description='Fração de Ejeção (%):', style=style, layout=layout_input),
    'Pressão Alta': widgets.ToggleButtons(options=[('Não', 0), ('Sim', 1)], description='Pressão Alta:', style=style, layout=layout_input),
    'Plaquetas': widgets.FloatText(value=265000, description='Plaquetas:', style=style, layout=layout_input),
    'Creatinina Sérica': widgets.FloatSlider(value=1.1, min=0.5, max=9.4, step=0.1, description='Creatinina Sérica:', style=style, layout=layout_input),
    'Sódio Sérico': widgets.IntSlider(value=137, min=110, max=150, description='Sódio Sérico:', style=style, layout=layout_input),
    'Sexo': widgets.ToggleButtons(options=[('Feminino', 0), ('Masculino', 1)], description='Sexo:', style=style, layout=layout_input),
    'Fumante': widgets.ToggleButtons(options=[('Não', 0), ('Sim', 1)], description='Fumante:', style=style, layout=layout_input),
    'Tempo': widgets.IntSlider(value=100, min=4, max=285, description='Tempo Acompanhamento:', style=style, layout=layout_input)
}

painel_esquerdo = widgets.VBox(list(inputs.values()), layout=widgets.Layout(padding='20px', border='1px solid #eee', width='42%', border_radius='10px'))
output_display = widgets.Output(layout=widgets.Layout(padding='20px', border='1px solid #eee', width='56%', min_height='500px', border_radius='10px'))

btn_analisar = widgets.Button(
    description=" RODAR ANÁLISE PREDITIVA", 
    layout=widgets.Layout(width='100%', height='55px', margin='15px 0')
)
btn_analisar.add_class("btn-analisar") # Aplica a cor verde via CSS

def processar(b):
    with output_display:
        clear_output(wait=True)
        dados_paciente = {label: widget.value for label, widget in inputs.items()}
        try:
            analise_interativa_de_risco(dados_paciente)
        except Exception as e:
            print(f" Erro ao processar dados: {e}")

btn_analisar.on_click(processar)

corpo = widgets.HBox([painel_esquerdo, output_display], layout=widgets.Layout(align_items='flex-start'))
dashboard = widgets.VBox([widgets.HTML(css_clinico), header, btn_analisar, corpo])

display(dashboard)